### Imports and Config

In [1]:
# ==========================================================
# 1. Imports and repo setup
# ==========================================================

import sys
from pathlib import Path
from datetime import date, timedelta
from calendar import monthrange

import pandas as pd

HERE = Path().resolve()
PROJECT_ROOT = HERE.parents[0]
sys.path.insert(0, str(PROJECT_ROOT))

from prm_opt.config import PlanningToggles, VEHICLE_MODELS

from prm_opt.optimise_prm_fleet_v2 import OptimiserConfig
from prm_opt.run_s25_s26_v2 import run_s25_s26_v2

from prm_opt.build_assumptions_v2 import build_assumptions_v2
from prm_opt.build_s26_assumptions import build_s26_assumptions

from prm_opt.ingest_s25 import ingest_s25
from prm_opt.ingest_s26 import ingest_s26
from prm_opt.build_jobs import build_jobs
from prm_opt.policy_s1 import apply_policy_s1
from prm_opt.outputs import baseline_s1_vehicle_curves_capacity, baseline_s1_summary

from prm_opt.ingest_s25_v2 import ingest_s25_v2
from prm_opt.ingest_s26_v2 import ingest_s26_v2
from prm_opt.build_flights_v2 import build_flights_v2

from prm_opt.policy_s2_flight_rules import run_s2_flight_rules

### User Parameters

In [2]:
# ==========================================================
# 2. User parameters
# ==========================================================

OUTPUT_ROOT = Path("outputs/prm_w26")
MONTHLY_OUTPUT_DIR = OUTPUT_ROOT / "monthly"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MONTHLY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STAND_PLAN_FILES = [
    "data/stands/stand_allocation-october.csv",
    "data/stands/stand_allocation-november.csv",
    "data/stands/stand_allocation-december.csv",
    "data/stands/stand_allocation-january.csv",
    "data/stands/stand_allocation-february.csv",
    "data/stands/stand_allocation-march.csv",
]

MONTHS_TO_RUN = [
    (2026, 10),
    (2026, 11),
    (2026, 12),
    (2027, 1),
    (2027, 2),
    (2027, 3),
]

ASSUMPTION_S25_START = "2025-07-01"
ASSUMPTION_S25_END = "2026-07-01"

PENETRATION_UPLIFT = 1.08

RUN_P100 = True
RUN_P90 = True

S2_AMB_SEATCAP = 7
S2_AMB_WCCAP = 1

S2_MINI_SEATCAP = 6
S2_MINI_WCCAP = 2

S2_MINIBUS_THRESHOLD_PRMS = 3

TIME_FREQ = "5min"



### Toggles and V2 config

In [3]:
# ==========================================================
# 3. Scenario configs
# ==========================================================

toggles_p100 = PlanningToggles(
    demand_mode="p100",
)

toggles_p90 = PlanningToggles(
    demand_mode="p90_stratified",
    p90_quantile=0.90,
    p90_strata=("SSR Code", "Has Own Chair", "A/D"),
)

v2_config = OptimiserConfig(
    tau_amb_solo_mins=31,
    tau_amb_comb_mins=26,
    tau_mini_mins=23,
    tau_push_mins=20,

    handover_buffer_mins=5,

    arrival_sla_target_pct=0.98,
    arrival_sla_target_mins=20,

    boarding_offset_mins=40,

    staff_amb=2,
    staff_mini=2,
    staff_per_prm_push=1,

    max_future_copies_per_model=5,

    solver_relative_gap=0.005,
)

### date splitting

In [4]:
# ==========================================================
# 4. Date helpers
# ==========================================================

def get_month_run_flags(year, month, *, today=None):
    """
    Historical = before today, exclusive of today.
    Future = from today, inclusive of today.

    All end dates are exclusive because the SQL queries use:
        >= start
        < end
    """

    if today is None:
        today = date.today()

    month_start = date(year, month, 1)

    month_end_inclusive = date(
        year,
        month,
        monthrange(year, month)[1],
    )

    next_month_start = month_end_inclusive + timedelta(days=1)

    kwargs = {}

    run_s25 = False
    run_s26 = False

    if month_end_inclusive < today:

        run_s25 = True
        kwargs["s25_start"] = str(month_start)
        kwargs["s25_end"] = str(next_month_start)

    elif month_start >= today:

        run_s26 = True
        kwargs["s26_start"] = str(month_start)
        kwargs["s26_end"] = str(next_month_start)

    else:

        run_s25 = True
        run_s26 = True

        kwargs["s25_start"] = str(month_start)
        kwargs["s25_end"] = str(today)

        kwargs["s26_start"] = str(today)
        kwargs["s26_end"] = str(next_month_start)

    return {
        "month_start": month_start,
        "month_end_inclusive": month_end_inclusive,
        "next_month_start": next_month_start,
        "run_s25": run_s25,
        "run_s26": run_s26,
        "kwargs": kwargs,
    }

### assumption caches

In [5]:
# ==========================================================
# 5. Assumption caches
# ==========================================================

_rule_assumptions_cache = None
_v2_assumptions_cache = None


def get_rule_assumptions():
    """
    Old S1/S1-style stack assumptions for future passenger-level ingest.
    """

    global _rule_assumptions_cache

    if _rule_assumptions_cache is None:
        _rule_assumptions_cache = build_s26_assumptions(
            s25_start=ASSUMPTION_S25_START,
            s25_end=ASSUMPTION_S25_END,
            bucket="15min",
        )

    return _rule_assumptions_cache


def get_v2_assumptions():
    """
    V2 assumptions for future flight-level ingest.
    """

    global _v2_assumptions_cache

    if _v2_assumptions_cache is None:
        _v2_assumptions_cache = build_assumptions_v2(
            s25_start=ASSUMPTION_S25_START,
            s25_end=ASSUMPTION_S25_END,
            config=v2_config,
            stand_plan_files=STAND_PLAN_FILES
        )

    return _v2_assumptions_cache


def apply_penetration_uplift_to_rule_assumptions(
    assumptions,
    uplift,
):
    """
    Apply uplift to old S1/S1-style future penetration assumptions.
    """

    out = {
        "inputs": assumptions["inputs"].copy(),
        "extras": assumptions.get("extras", {}).copy(),
    }

    pen = out["inputs"]["penetration_rates"].copy()

    pen["penetration_base"] = pen["penetration"]
    pen["penetration_uplift"] = float(uplift)
    pen["penetration"] = pen["penetration"] * float(uplift)

    out["inputs"]["penetration_rates"] = pen

    return out

### S1 runner

In [6]:
# ==========================================================
# 7. S1 runner
# ==========================================================

def build_s1_jobs_for_period(
    *,
    month_info,
    toggles,
    rule_assumptions=None,
    seed=42,
):

    run_s25 = month_info["run_s25"]
    run_s26 = month_info["run_s26"]
    kwargs = month_info["kwargs"]

    parts = []

    if run_s25:

        df_hist = ingest_s25(
            start=kwargs["s25_start"],
            end=kwargs["s25_end"],
            seed=seed,
        )

        jobs_hist = build_jobs(
            df_hist,
            bucket="15min",
            toggles=toggles,
            use_90th_percentile_cap=False,
        )

        jobs_hist["source_period"] = "S25"
        parts.append(jobs_hist)

    if run_s26:

        if rule_assumptions is None:
            raise ValueError("Future S1 requires rule_assumptions.")

        df_future = ingest_s26(
            start=kwargs["s26_start"],
            end=kwargs["s26_end"],
            **rule_assumptions["inputs"],
            seed=seed,
        )

        jobs_future = build_jobs(
            df_future,
            bucket="15min",
            toggles=toggles,
            use_90th_percentile_cap=False,
        )

        jobs_future["source_period"] = "S26"
        parts.append(jobs_future)

    if len(parts) == 0:
        return pd.DataFrame()

    jobs = pd.concat(
        parts,
        ignore_index=True,
    )

    jobs.index.name = "j"

    return jobs


def _current_fleet_counts_from_vehicle_models():

    current_amb = sum(
        1
        for _, v in VEHICLE_MODELS.items()
        if v["type"] == "Amb"
        and not bool(v.get("is_future", False))
    )

    current_mini = sum(
        1
        for _, v in VEHICLE_MODELS.items()
        if v["type"] == "Mini"
        and not bool(v.get("is_future", False))
    )

    return current_amb, current_mini


def run_s1_for_period(
    *,
    month_info,
    toggles,
    rule_assumptions=None,
    seed=42,
):

    jobs = build_s1_jobs_for_period(
        month_info=month_info,
        toggles=toggles,
        rule_assumptions=rule_assumptions,
        seed=seed,
    )

    if jobs is None or len(jobs) == 0:

        empty_series = pd.Series(dtype=int)

        return {
            "jobs": pd.DataFrame(),
            "summary": {},
            "ambulift_curve": empty_series,
            "minibus_curve": empty_series,
            "driver_curve": empty_series,
            "veh_agent_curve": empty_series,
            "pusher_curve": empty_series,
            "fb_detail": pd.DataFrame(),
        }

    decisions = apply_policy_s1(jobs)

    jobs["s1_decision"] = jobs.index.map(decisions.get)

    curves = baseline_s1_vehicle_curves_capacity(
        jobs,
        decision_col="s1_decision",
        bucket_col="s",
        count_no_vehicle_as_push=True,
        amb_seatcap=3,
        amb_wccap=1,
        mini_seatcap=6,
        mini_wccap=2,
        duration_mins=20,
        time_freq=TIME_FREQ,
    )

    current_amb, current_mini = _current_fleet_counts_from_vehicle_models()

    summary = baseline_s1_summary(
        jobs,
        curves,
        current_amb=current_amb,
        current_mini=current_mini,
    )

    return {
        "jobs": jobs,
        "summary": summary,
        "ambulift_curve": curves["ambulift_curve"],
        "minibus_curve": curves["minibus_curve"],
        "driver_curve": curves["driver_curve"],
        "veh_agent_curve": curves["veh_agent_curve"],
        "pusher_curve": curves["pusher_curve"],
        "fb_detail": curves["fb_detail"],
    }

### build V2 passengers and flights for S2

In [7]:
# ==========================================================
# 8. Build V2 passengers and flights for S2
# ==========================================================

def build_v2_passengers_for_period(
    *,
    month_info,
    v2_assumptions=None,
    seed=42,
):

    run_s25 = month_info["run_s25"]
    run_s26 = month_info["run_s26"]
    kwargs = month_info["kwargs"]

    parts = []

    if run_s25:

        df_hist = ingest_s25_v2(
            start=kwargs["s25_start"],
            end=kwargs["s25_end"],
            seed=seed,
        )

        df_hist["source_period"] = "S25"
        parts.append(df_hist)

    if run_s26:

        if v2_assumptions is None:
            raise ValueError("Future V2/S2 requires v2_assumptions.")

        df_future = ingest_s26_v2(
            start=kwargs["s26_start"],
            end=kwargs["s26_end"],
            penetration_rates=v2_assumptions["penetration_rates"],
            ssr_mix=v2_assumptions["ssr_mix"],
            stand_actuals=v2_assumptions["stand_actuals"],
            stand_dist=v2_assumptions["stand_dist"],
            penetration_uplift=PENETRATION_UPLIFT,
            seed=seed,
        )

        df_future["source_period"] = "S26"
        parts.append(df_future)

    if len(parts) == 0:
        return pd.DataFrame()

    passengers = pd.concat(
        parts,
        ignore_index=True,
    )

    return passengers


def build_v2_flights_for_period(
    *,
    month_info,
    demand_mode,
    v2_assumptions=None,
    seed=42,
):

    passengers = build_v2_passengers_for_period(
        month_info=month_info,
        v2_assumptions=v2_assumptions,
        seed=seed,
    )

    if passengers is None or len(passengers) == 0:
        return pd.DataFrame()

    flights = build_flights_v2(
        passengers,
        demand_mode=demand_mode,
        p90_quantile=0.90,
        boarding_offset_mins=v2_config.boarding_offset_mins,
        p90_strata=("demand_category", "A/D"),
    )

    return flights

### S2 runner

In [8]:
# ==========================================================
# 9. S2 runner
# ==========================================================

def run_s2_for_period(
    *,
    month_info,
    demand_mode,
    v2_assumptions=None,
    seed=42,
):

    flights = build_v2_flights_for_period(
        month_info=month_info,
        demand_mode=demand_mode,
        v2_assumptions=v2_assumptions,
        seed=seed,
    )

    if flights is None or len(flights) == 0:

        empty_series = pd.Series(dtype=int)

        return {
            "flights": pd.DataFrame(),
            "flight_requirements": pd.DataFrame(),
            "summary": {},
            "ambulift_curve": empty_series,
            "minibus_curve": empty_series,
            "driver_curve": empty_series,
            "veh_agent_curve": empty_series,
            "event_detail": pd.DataFrame(),
            "curve_df": pd.DataFrame(),
        }

    out = run_s2_flight_rules(
        flights=flights,
        config=v2_config,
        amb_seatcap=S2_AMB_SEATCAP,
        amb_wccap=S2_AMB_WCCAP,
        mini_seatcap=S2_MINI_SEATCAP,
        mini_wccap=S2_MINI_WCCAP,
        minibus_threshold_prms=S2_MINIBUS_THRESHOLD_PRMS,
        time_freq=TIME_FREQ,
    )

    out["flights"] = flights

    return out

### V2 optimiser runner

In [9]:
# ==========================================================
# 10. V2 optimiser runner
# ==========================================================

def run_v2_optimizer_for_period(
    *,
    month_info,
    v2_assumptions=None,
    output_dir,
    output_xlsx_name,
):

    run_s25 = month_info["run_s25"]
    run_s26 = month_info["run_s26"]
    kwargs = month_info["kwargs"]

    call_kwargs = {
        "run_s25": run_s25,
        "run_s26": run_s26,

        "vehicle_models": VEHICLE_MODELS,
        "config": v2_config,

        "output_dir": str(output_dir),
        "output_xlsx_name": output_xlsx_name,

        "run_p100": True,
        "run_p90": True,

        "run_fleet_report": True,
        "fleet_report_policy": "if_needed",
        "current_fleet_only_first": True,

        "max_ev10": 5,
        "max_ev18": 5,

        **kwargs,
    }

    if run_s26:

        if v2_assumptions is None:
            raise ValueError(
                "Future V2 optimiser requires v2_assumptions."
            )

        call_kwargs.update(
            {
                "penetration_rates": v2_assumptions["penetration_rates"],
                "ssr_mix": v2_assumptions["ssr_mix"],
                "stand_actuals": v2_assumptions["stand_actuals"],
                "stand_dist": v2_assumptions["stand_dist"],
                "penetration_uplift": PENETRATION_UPLIFT,
            }
        )

    result = run_s25_s26_v2(**call_kwargs)

    return result

# workbook write and V2 summary helper

In [10]:
# ==========================================================
# 11. Workbook writer and V2 summary helper
# ==========================================================

def _safe_sheet_name(name):

    invalid = ["\\", "/", "*", "?", ":", "[", "]"]

    out = str(name)

    for ch in invalid:
        out = out.replace(ch, "_")

    return out[:31]


def _write_df(
    writer,
    obj,
    sheet_name,
):

    if obj is None:
        return

    if isinstance(obj, dict):
        df = pd.DataFrame([obj])

    elif isinstance(obj, pd.Series):
        df = obj.rename("value").reset_index()

    elif isinstance(obj, pd.DataFrame):
        df = obj

    else:
        df = pd.DataFrame(obj)

    df.to_excel(
        writer,
        sheet_name=_safe_sheet_name(sheet_name),
        index=False,
    )


def write_monthly_combined_workbook(
    *,
    output_path,
    month_rows,
    s1_outputs,
    s2_outputs,
    v2_result,
):

    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with pd.ExcelWriter(
        output_path,
        engine="openpyxl",
    ) as writer:

        _write_df(
            writer,
            pd.DataFrame(month_rows),
            "Monthly_Summary",
        )

        # ----------------------------
        # S1 outputs
        # ----------------------------

        for mode_key, s1 in s1_outputs.items():

            _write_df(
                writer,
                s1.get("summary"),
                f"S1_{mode_key}_Summary",
            )

            _write_df(
                writer,
                s1.get("jobs"),
                f"S1_{mode_key}_Jobs",
            )

            _write_df(
                writer,
                s1.get("fb_detail"),
                f"S1_{mode_key}_FB_Detail",
            )

            _write_df(
                writer,
                s1.get("ambulift_curve"),
                f"S1_{mode_key}_Amb_Curve",
            )

            _write_df(
                writer,
                s1.get("minibus_curve"),
                f"S1_{mode_key}_Mini_Curve",
            )

        # ----------------------------
        # S2 outputs
        # ----------------------------

        for mode_key, s2 in s2_outputs.items():

            _write_df(
                writer,
                s2.get("summary"),
                f"S2_{mode_key}_Summary",
            )

            _write_df(
                writer,
                s2.get("flights"),
                f"S2_{mode_key}_Flights",
            )

            _write_df(
                writer,
                s2.get("flight_requirements"),
                f"S2_{mode_key}_Requirements",
            )

            _write_df(
                writer,
                s2.get("curve_df"),
                f"S2_{mode_key}_Curve",
            )

            _write_df(
                writer,
                s2.get("event_detail"),
                f"S2_{mode_key}_Events",
            )

        # ----------------------------
        # V2 outputs
        # ----------------------------

        if v2_result is not None:

            if "outputs_p100" in v2_result:

                for key, df in v2_result["outputs_p100"].items():

                    if isinstance(df, pd.DataFrame):

                        _write_df(
                            writer,
                            df,
                            f"V2_P100_{key}",
                        )

            if "outputs_p90" in v2_result:

                for key, df in v2_result["outputs_p90"].items():

                    if isinstance(df, pd.DataFrame):

                        _write_df(
                            writer,
                            df,
                            f"V2_P90_{key}",
                        )
            
            extra_v2_keys = [
                "p100_base_check",
                "p90_base_check",
                "fleet_requirements_p100",
                "fleet_requirements_p90",
                "recommended_p100",
                "recommended_p90",
            ]

            for key in extra_v2_keys:

                if (
                    key in v2_result
                    and isinstance(v2_result[key], pd.DataFrame)
                ):

                    _write_df(
                        writer,
                        v2_result[key],
                        f"V2_{key}",
                    )

    return output_path


def extract_v2_month_summary(
    v2_result,
    mode_key,
):

    if v2_result is None:
        return {}

    if mode_key == "P100":
        outputs = v2_result.get("outputs_p100", {})
    else:
        outputs = v2_result.get("outputs_p90", {})

    sla = outputs.get(
        "sla_summary",
        pd.DataFrame(),
    )

    fleet = outputs.get(
        "fleet_utilisation",
        pd.DataFrame(),
    )

    arrival_sla_pct = None

    if isinstance(sla, pd.DataFrame) and len(sla) > 0:

        row = sla.iloc[0]

        arrival_sla_pct = row.get(
            "arrival_sla_pct"
        )

    peak_amb = None
    peak_mini = None

    if (
        isinstance(fleet, pd.DataFrame)
        and len(fleet) > 0
        and "vehicle_type" in fleet.columns
        and "required_for_schedule" in fleet.columns
    ):

        peak_amb = int(
            pd.to_numeric(
                fleet.loc[
                    fleet["vehicle_type"] == "Amb",
                    "required_for_schedule",
                ],
                errors="coerce",
            ).fillna(0).sum()
        )

        peak_mini = int(
            pd.to_numeric(
                fleet.loc[
                    fleet["vehicle_type"] == "Mini",
                    "required_for_schedule",
                ],
                errors="coerce",
            ).fillna(0).sum()
        )

    return {
        "V2_arrival_sla_pct": arrival_sla_pct,
        "V2_PeakAmb": peak_amb,
        "V2_PeakMini": peak_mini,
    }

### run one month

In [11]:
# ==========================================================
# 12. Run one month
# ==========================================================

def run_combined_month(
    year,
    month,
    *,
    seed=42,
):

    month_info = get_month_run_flags(
        year,
        month,
    )

    month_start = month_info["month_start"]

    month_tag = f"{year}_{month:02d}"

    print("")
    print("=" * 80)
    print(f"RUNNING COMBINED MONTH: {month_tag}")
    print("=" * 80)

    needs_future = bool(
        month_info["run_s26"]
    )

    rule_assumptions = None
    v2_assumptions = None

    if needs_future:

        rule_assumptions = (
            apply_penetration_uplift_to_rule_assumptions(
                get_rule_assumptions(),
                PENETRATION_UPLIFT,
            )
        )

        v2_assumptions = (
            get_v2_assumptions()
        )

    s1_outputs = {}
    s2_outputs = {}

    modes_to_run = []

    if RUN_P100:
        modes_to_run.append(
            (
                "P100",
                "p100",
                toggles_p100,
            )
        )

    if RUN_P90:
        modes_to_run.append(
            (
                "P90",
                "p90",
                toggles_p90,
            )
        )

    for mode_key, demand_mode, toggles in modes_to_run:

        print("")
        print(f"Running S1 {mode_key}")

        s1_outputs[mode_key] = (
            run_s1_for_period(
                month_info=month_info,
                toggles=toggles,
                rule_assumptions=rule_assumptions,
                seed=seed,
            )
        )

        print(f"Running S2 {mode_key}")

        s2_outputs[mode_key] = (
            run_s2_for_period(
                month_info=month_info,
                demand_mode=demand_mode,
                v2_assumptions=v2_assumptions,
                seed=seed,
            )
        )

    print("")
    print("Running V2 optimiser")

    v2_result = run_v2_optimizer_for_period(
        month_info=month_info,
        v2_assumptions=v2_assumptions,
        output_dir=MONTHLY_OUTPUT_DIR,
        output_xlsx_name=f"{month_tag}_v2.xlsx",
    )

    month_rows = []

    for mode_key, _, _ in modes_to_run:

        s1_summary = (
            s1_outputs[mode_key]
            .get("summary", {})
        )

        s2_summary = (
            s2_outputs[mode_key]
            .get("summary", {})
        )

        v2_summary = (
            extract_v2_month_summary(
                v2_result,
                mode_key,
            )
        )

        row = {

            "S1_PeakAmb":
                s1_summary.get("PeakAmb"),

            "S1_PeakMini":
                s1_summary.get("PeakMini"),

            "S1_PeakDrivers":
                s1_summary.get("PeakDrivers"),

            "S1_PeakVehAgents":
                s1_summary.get("PeakVehAgents"),

            "S2_PeakAmb":
                s2_summary.get("PeakAmb"),

            "S2_PeakMini":
                s2_summary.get("PeakMini"),

            "S2_PeakDrivers":
                s2_summary.get("PeakDrivers"),

            "S2_PeakVehAgents":
                s2_summary.get("PeakVehAgents"),

            **v2_summary,
        }

        month_rows.append(row)

    combined_path = (
        MONTHLY_OUTPUT_DIR
        / f"{month_tag}_combined.xlsx"
    )

    write_monthly_combined_workbook(
        output_path=combined_path,
        month_rows=month_rows,
        s1_outputs=s1_outputs,
        s2_outputs=s2_outputs,
        v2_result=v2_result,
    )

    print("")
    print(f"Completed month: {month_tag}")
    print(f"Workbook: {combined_path}")

    return {

        "month_rows":
            month_rows,

        "s1_outputs":
            s1_outputs,

        "s2_outputs":
            s2_outputs,

        "v2_result":
            v2_result,

        "combined_workbook_path":
            combined_path,
    }

### Test Call

In [12]:
october_result = run_combined_month(
    2026,
    10,
)

pd.DataFrame(october_result["month_rows"])


RUNNING COMBINED MONTH: 2026_10

Unmatched passenger rows after merge (missing Chocks DT): 7451
Unique unmatched flight keys: 4434

Unmatched key reasons:
reason
no_flight_candidate               3551
scheduled_dt_mismatch              729
exact_match_should_have_joined     154
Name: count, dtype: int64

Dropped 7100 passenger rows due to reasons {'no_flight_candidate', 'scheduled_dt_mismatch'}
[1/5] Penetration + SSR mix…
    ✓ penetration + SSR mix built   [242.81s]
[2/5] Service times…
[BUILD_JOBS] Dropping 66 jobs already impossible at release (beyond SLA + max late)

[BUILD_JOBS] Dropping 4 arrival jobs with expired SLA window (max_late_mins=180).
       Passenger ID Flight Number Airline Code      sla_start_time  \
53000      13159410           748           LS 2025-09-30 01:35:15   
53001      13159411           748           LS 2025-09-30 01:35:15   
53004      13159414           748           LS 2025-09-30 01:35:15   
53005      13159415           748           LS 2025-09-30 

,S1_PeakAmb,S1_PeakMini,S1_PeakDrivers,S1_PeakVehAgents,S2_PeakAmb,S2_PeakMini,S2_PeakDrivers,S2_PeakVehAgents,V2_arrival_sla_pct,V2_PeakAmb,V2_PeakMini
0,10,2,11,11,11,16,25,25,99.806103,11,3
1,11,2,11,11,11,14,23,23,99.924528,12,3


In [13]:
november_result = run_combined_month(
    2026,
    11,
)

pd.DataFrame(november_result["month_rows"])


RUNNING COMBINED MONTH: 2026_11

Running S1 P100

[DEBUG A] Future flights loaded & Pax computed
df_flights rows: 8920
Airlines (sample): ['KL' 'AF' 'FR' 'U2' 'RK' 'LH' 'BA' 'LS' 'EC' 'UA']
Countries (sample): ['NETHERLANDS' 'FRANCE' 'IRISH REPUBLIC' 'PORTUGAL' 'SPAIN' 'POLAND'
 'ALBANIA' 'GERMANY' 'GREAT BRITAIN' 'ITALY']

Pax describe:
count    8920.000000
mean      144.445964
std        56.776957
min         0.000000
25%       118.000000
50%       159.000000
75%       180.000000
max       312.000000
Name: Pax, dtype: float64

Top 10 rows (Airline, CountryName, Sector, dir, Pax):
   Airline     CountryName         Sector dir    Pax
0       KL     NETHERLANDS  International   D  227.0
1       AF          FRANCE  International   D  140.0
2       FR  IRISH REPUBLIC            CTA   D  191.0
3       FR        PORTUGAL  International   D  186.0
4       FR  IRISH REPUBLIC            CTA   D  191.0
5       U2          FRANCE  International   D  179.0
6       FR           SPAIN  Internation

,S1_PeakAmb,S1_PeakMini,S1_PeakDrivers,S1_PeakVehAgents,S2_PeakAmb,S2_PeakMini,S2_PeakDrivers,S2_PeakVehAgents,V2_arrival_sla_pct,V2_PeakAmb,V2_PeakMini
0,10,2,10,10,10,11,19,19,99.769884,12,3
1,10,2,10,10,10,11,19,19,99.920102,12,3


In [14]:
december_result = run_combined_month(
    2026,
    12,
)

pd.DataFrame(december_result["month_rows"])


RUNNING COMBINED MONTH: 2026_12

Running S1 P100

[DEBUG A] Future flights loaded & Pax computed
df_flights rows: 9150
Airlines (sample): ['FR' 'U2' 'BA' 'FI' 'AF' 'A3' 'EA' 'DY' 'LM' 'KL']
Countries (sample): ['SPAIN' 'GREAT BRITAIN' 'GERMANY' 'ICELAND' 'FRANCE' 'EGYPT' 'GREECE'
 'IRISH REPUBLIC' 'NORWAY' 'NETHERLANDS']

Pax describe:
count    9150.000000
mean      144.236503
std        54.975747
min         0.000000
25%       118.000000
50%       157.500000
75%       180.000000
max       312.000000
Name: Pax, dtype: float64

Top 10 rows (Airline, CountryName, Sector, dir, Pax):
   Airline     CountryName         Sector dir    Pax
0       FR           SPAIN  International   D   85.0
1       U2   GREAT BRITAIN       Domestic   D  146.0
2       BA   GREAT BRITAIN       Domestic   D  204.0
3       U2         GERMANY  International   D  235.0
4       FI         ICELAND  International   D  160.0
5       AF          FRANCE  International   D  131.0
6       U2           EGYPT  International

,S1_PeakAmb,S1_PeakMini,S1_PeakDrivers,S1_PeakVehAgents,S2_PeakAmb,S2_PeakMini,S2_PeakDrivers,S2_PeakVehAgents,V2_arrival_sla_pct,V2_PeakAmb,V2_PeakMini
0,10,2,10,10,11,11,21,21,99.864171,13,3
1,9,1,9,9,11,11,21,21,99.950650,12,3


In [15]:
january_result = run_combined_month(
    2027,
    1,
)

pd.DataFrame(january_result["month_rows"])


RUNNING COMBINED MONTH: 2027_01

Running S1 P100

[DEBUG A] Future flights loaded & Pax computed
df_flights rows: 8099
Airlines (sample): ['VY' 'FR' 'U2' 'RK' 'SK' 'LM' 'BA' 'UA' 'AF' 'DY']
Countries (sample): ['SPAIN' 'POLAND' 'AUSTRIA' 'GREAT BRITAIN' 'CZECH REPUBLIC' 'DENMARK'
 'GERMANY' 'UNITED STATES OF AMERICA' 'FRANCE' 'NORWAY']

Pax describe:
count    8099.000000
mean      142.145820
std        57.164454
min         0.000000
25%       112.000000
50%       156.000000
75%       180.000000
max       312.000000
Name: Pax, dtype: float64

Top 10 rows (Airline, CountryName, Sector, dir, Pax):
   Airline     CountryName         Sector dir    Pax
0       VY           SPAIN  International   A  186.0
1       FR          POLAND  International   A  175.0
2       FR         AUSTRIA  International   A  195.0
3       U2   GREAT BRITAIN       Domestic   A  120.0
4       FR  CZECH REPUBLIC  International   D  177.0
5       RK   GREAT BRITAIN       Domestic   D  187.0
6       FR          POLAND

,S1_PeakAmb,S1_PeakMini,S1_PeakDrivers,S1_PeakVehAgents,S2_PeakAmb,S2_PeakMini,S2_PeakDrivers,S2_PeakVehAgents,V2_arrival_sla_pct,V2_PeakAmb,V2_PeakMini
0,11,2,12,12,13,12,23,23,99.855538,14,3
1,9,1,9,9,13,12,23,23,99.964126,13,3


In [16]:
february_result = run_combined_month(
    2027,
    2,
)

pd.DataFrame(february_result["month_rows"])


RUNNING COMBINED MONTH: 2027_02

Running S1 P100

[DEBUG A] Future flights loaded & Pax computed
df_flights rows: 7784
Airlines (sample): ['FR' 'LS' 'U2' 'RK' 'LM' 'VY' 'BA' 'EA' 'EC' 'DS']
Countries (sample): ['BELGIUM' 'PORTUGAL' 'NORWAY' 'GREAT BRITAIN' 'GREECE' 'SPAIN' 'FRANCE'
 'IRISH REPUBLIC' 'SWITZERLAND' 'AUSTRIA']

Pax describe:
count    7784.000000
mean      145.576953
std        56.478301
min         0.000000
25%       117.000000
50%       160.000000
75%       181.000000
max       312.000000
Name: Pax, dtype: float64

Top 10 rows (Airline, CountryName, Sector, dir, Pax):
   Airline    CountryName         Sector dir    Pax
0       FR        BELGIUM  International   A  170.0
1       LS       PORTUGAL  International   D  198.0
2       U2         NORWAY  International   D  152.0
3       RK  GREAT BRITAIN       Domestic   D  182.0
4       LM  GREAT BRITAIN       Domestic   A   13.0
5       U2  GREAT BRITAIN       Domestic   A  181.0
6       U2         GREECE  International   D 

,S1_PeakAmb,S1_PeakMini,S1_PeakDrivers,S1_PeakVehAgents,S2_PeakAmb,S2_PeakMini,S2_PeakDrivers,S2_PeakVehAgents,V2_arrival_sla_pct,V2_PeakAmb,V2_PeakMini
0,9,2,9,9,11,11,21,21,99.768058,13,3
1,9,1,9,9,11,10,20,20,99.833887,12,3


In [12]:
march_result = run_combined_month(
    2027,
    3,
)

pd.DataFrame(march_result["month_rows"])


RUNNING COMBINED MONTH: 2027_03

Unmatched passenger rows after merge (missing Chocks DT): 7451
Unique unmatched flight keys: 4434

Unmatched key reasons:
reason
no_flight_candidate               3551
scheduled_dt_mismatch              729
exact_match_should_have_joined     154
Name: count, dtype: int64

Dropped 7100 passenger rows due to reasons {'scheduled_dt_mismatch', 'no_flight_candidate'}
[1/5] Penetration + SSR mix…
    ✓ penetration + SSR mix built   [675.81s]
[2/5] Service times…
[BUILD_JOBS] Dropping 66 jobs already impossible at release (beyond SLA + max late)

[BUILD_JOBS] Dropping 4 arrival jobs with expired SLA window (max_late_mins=180).
       Passenger ID Flight Number Airline Code      sla_start_time  \
53000      13159410           748           LS 2025-09-30 01:35:15   
53001      13159411           748           LS 2025-09-30 01:35:15   
53004      13159414           748           LS 2025-09-30 01:35:15   
53005      13159415           748           LS 2025-09-30 

,S1_PeakAmb,S1_PeakMini,S1_PeakDrivers,S1_PeakVehAgents,S2_PeakAmb,S2_PeakMini,S2_PeakDrivers,S2_PeakVehAgents,V2_arrival_sla_pct,V2_PeakAmb,V2_PeakMini
0,11,1,11,11,10,12,20,20,99.833528,11,3
1,9,1,9,9,10,11,20,20,99.889827,13,3
